# Forward/Reverse Similarity & Ion Ratio Analysis

## Corrected Analysis (April 2026)

This notebook tests whether **forward/reverse/ion-ratio spectral metrics** improve the annotation confidence model (baseline CV AUC = 0.837).

### Five Corrections from Previous Version

1. **Removed 0.75 threshold** — Previously filtered hits before computing forward/reverse (survivor bias). Now: compute on ALL hits with `library_wiki_id`.

2. **Added entropy weighting** — Forward/reverse now use `ms_entropy.apply_weight_to_intensity()` preprocessing, matching internal `calculate_entropy_similarity()` logic.

3. **Corrected entropy formula** — Validated that unmatched peaks reduce similarity (appear at half intensity in merged spectrum).

4. **Include MB-EU** — Removed from EXCLUDED_DBS (it's public, not in-house). Only exclude `5min_hilic_neg` and `5min_lipid_neg`.

5. **Compound deduplication** — Deduplicate by `(wiki_id, hit_ik14)` keeping best `entropy_similarity`, removing 8.5× duplication from same compound in multiple DBs.

In [5]:
import os, json, time, requests, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.metrics import roc_auc_score
import ms_entropy
warnings.filterwarnings('ignore')

SPECTRA_PATH    = '../data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx'
HITS_PATH       = '../data/orbitrap_hits_refetched.csv'
SOLID_TP_PATH   = '../data/solid_tp.csv'
INCHIKEY_PATH   = '../data/inchikey_cache.json'
LIB_CACHE       = '../data/library_peaks_cache.json'
QUERY_CACHE     = '../data/query_peaks_cache.json'
LIB_URL         = 'https://masswiki.us-west-2.elasticbeanstalk.com/reference_library/get_spectra_data'
BATCH_SIZE      = 50
PPM_TOL         = 10.0
SIM_GAP_THRESHOLD = 0.75
EXCLUDED_DBS    = {'5min_hilic_neg', '5min_lipid_neg'}
print('✓ Imports loaded')

✓ Imports loaded


In [ ]:
# Load spectra + labels
spectra = pd.read_excel(SPECTRA_PATH, header=4)
spectra['label'] = 'unlabeled'
spectra.loc[:1297, 'label'] = 'TP'
spectra.loc[spectra['name'].str.startswith('yy_', na=False), 'label'] = 'FP'
spectra['holdout'] = spectra['wiki_id'].isin(set(pd.read_csv(SOLID_TP_PATH)['wiki_id']))

# Load IK14 cache (values are dicts: {'ik14': '...', 'inchikey': '...'})
with open(INCHIKEY_PATH) as f:
    inchikey_cache = json.load(f)

def get_ik14(smiles):
    """Extract IK14 string from inchikey_cache."""
    if not smiles or not isinstance(smiles, str):
        return ''
    val = inchikey_cache.get(smiles, '')
    if isinstance(val, dict):
        return val.get('ik14', '')
    return val if isinstance(val, str) else ''

# Get annotation IK14 from the spectra's smiles column
spectra['anno_ik14'] = spectra['smiles'].fillna('').apply(get_ik14)

ann = spectra[spectra['label'].isin(['TP','FP'])][['wiki_id','name','label','holdout','anno_ik14']].copy()
print(f'Annotated spectra: {len(ann):,}')
print(f'  With anno_ik14: {(ann["anno_ik14"] != "").sum():,}')

# Load hits and map IK14
hits = pd.read_csv(HITS_PATH)
hits['hit_ik14'] = hits['smiles'].fillna('').apply(get_ik14)
print(f'Loaded {len(hits):,} hits')
print(f'  With hit_ik14: {(hits["hit_ik14"] != "").sum():,}  ({hits[hits["hit_ik14"]!=""]["hit_ik14"].nunique():,} unique)')

# Join and filter
joint = hits.merge(ann, on='wiki_id', how='inner')
joint_f = joint[
    ~joint['db'].isin(EXCLUDED_DBS) &
    joint['library_wiki_id'].notna()
].copy()

# Hit-level correctness: TP spectrum AND hit IK14 matches annotation IK14
joint_f['hit_correct'] = (
    (joint_f['label'] == 'TP') &
    (joint_f['hit_ik14'] != '') &
    (joint_f['anno_ik14'] != '') &
    (joint_f['hit_ik14'] == joint_f['anno_ik14'])
)

# Is this the annotation compound's hit?
joint_f['is_anno_hit'] = (
    (joint_f['hit_ik14'] != '') &
    (joint_f['anno_ik14'] != '') &
    (joint_f['hit_ik14'] == joint_f['anno_ik14'])
)

print(f'After DB filter + library_wiki_id: {len(joint_f):,} hits')
print(f'  Correct (TP + IK14 match): {joint_f["hit_correct"].sum():,}')
print(f'  Annotation hits (IK14 match, any label): {joint_f["is_anno_hit"].sum():,}')
print(f'\\nDB breakdown:')
print(joint_f['db'].value_counts())

In [7]:
# FIX #5: Deduplicate by (wiki_id, hit_ik14)
before_dedup = len(joint_f)

joint_f['dedup_key'] = joint_f.apply(
    lambda r: (r['wiki_id'], r['hit_ik14']) if r['hit_ik14'] else (r['wiki_id'], r['lib_name']),
    axis=1
)

joint_dedup = (
    joint_f
    .sort_values('entropy_similarity', ascending=False)
    .drop_duplicates(subset='dedup_key')
    .reset_index(drop=True)
)

print(f'Deduplication:')
print(f'  Before: {before_dedup:,}')
print(f'  After: {len(joint_dedup):,}')
print(f'  Removed: {before_dedup - len(joint_dedup):,} ({(before_dedup-len(joint_dedup))/before_dedup:.1%})')
print()
print(f'Using hit_ik14: {(joint_dedup["hit_ik14"] != "").sum():,}')
print(f'Using lib_name: {(joint_dedup["hit_ik14"] == "").sum():,}')

Deduplication:
  Before: 107,875
  After: 14,002
  Removed: 93,873 (87.0%)

Using hit_ik14: 12,622
Using lib_name: 1,380


In [8]:
# Load and fetch library peaks
lib_cache = {}
if os.path.exists(LIB_CACHE):
    with open(LIB_CACHE) as f:
        lib_cache = json.load(f)
    print(f'Loaded library cache: {len(lib_cache):,} entries')

unique_lib_ids = joint_dedup['library_wiki_id'].dropna().unique().tolist()
to_fetch = [lid for lid in unique_lib_ids if lid not in lib_cache]
print(f'To fetch: {len(to_fetch):,}  (already cached: {len(unique_lib_ids)-len(to_fetch):,})')

if len(to_fetch) > 0:
    print('Fetching missing library peaks...')
    def fetch_library_batch(id_list):
        resp = requests.post(
            LIB_URL,
            json={'id_list': id_list, 'get_details': False, 'include_fields': ['peaks']},
            headers={'accept': 'application/json', 'Content-Type': 'application/json'},
            timeout=30
        )
        return resp.json() if resp.status_code == 200 else []

    for i in range(0, len(to_fetch), BATCH_SIZE):
        batch = to_fetch[i:i+BATCH_SIZE]
        for entry in fetch_library_batch(batch):
            if entry.get('wiki_id') and entry.get('peaks'):
                lib_cache[entry['wiki_id']] = entry['peaks']
        if i % 500 == 0 and i > 0:
            print(f'  {i:,}/{len(to_fetch):,}...')
            with open(LIB_CACHE, 'w') as f:
                json.dump(lib_cache, f)
        time.sleep(0.05)

with open(LIB_CACHE, 'w') as f:
    json.dump(lib_cache, f)

covered = sum(1 for lid in unique_lib_ids if lid in lib_cache)
print(f'Coverage: {covered:,}/{len(unique_lib_ids):,} ({covered/len(unique_lib_ids):.1%})')

Loaded library cache: 26,132 entries
To fetch: 883  (already cached: 10,589)
Fetching missing library peaks...
  500/883...
Coverage: 11,472/11,472 (100.0%)


In [9]:
# Load query peaks (fully populated by refetch_orbitrap_hits.py)
query_cache = {}
if os.path.exists(QUERY_CACHE):
    with open(QUERY_CACHE) as f:
        query_cache = json.load(f)
    print(f'Loaded query cache: {len(query_cache):,} entries')
else:
    print(f'⚠️  Query cache not found')

Loaded query cache: 1,513 entries


In [10]:
# FIX #3: Verify entropy similarity recomputation
sample_size = min(200, len(joint_dedup[joint_dedup['entropy_similarity'].notna()]))
sample_hits = joint_dedup[joint_dedup['entropy_similarity'].notna()].sample(n=sample_size, random_state=42)

diffs = []
for idx, row in sample_hits.iterrows():
    wid = row['wiki_id']
    lid = row['library_wiki_id']
    
    q_peaks = query_cache.get(wid)
    l_peaks = lib_cache.get(lid)
    
    if q_peaks and l_peaks:
        try:
            recomputed_sim = ms_entropy.calculate_entropy_similarity(q_peaks, l_peaks)
            diffs.append(abs(row['entropy_similarity'] - recomputed_sim))
        except:
            pass

diffs = np.array(diffs)
print(f'Entropy similarity verification ({len(diffs)}/{sample_size}):' )
print(f'  Mean diff: {diffs.mean():.6f}')
print(f'  Median diff: {np.median(diffs):.6f}')
print(f'  Max diff: {diffs.max():.6f}')
print(f'  % within 0.01: {(diffs < 0.01).sum() / len(diffs) * 100:.1f}%')

Entropy similarity verification (200/200):
  Mean diff: 0.038953
  Median diff: 0.010166
  Max diff: 0.293260
  % within 0.01: 50.0%


In [ ]:
# FIX #2: Compute forward/reverse with entropy weighting
def compute_scores(query_peaks_raw, lib_peaks_raw, ppm_tol=PPM_TOL):
    if not query_peaks_raw or not lib_peaks_raw:
        return None
    
    try:
        # Clean and weight both spectra
        q_clean = ms_entropy.clean_spectrum(query_peaks_raw)
        l_clean = ms_entropy.clean_spectrum(lib_peaks_raw)
        q_weighted = ms_entropy.apply_weight_to_intensity(q_clean)
        l_weighted = ms_entropy.apply_weight_to_intensity(l_clean)
        
        q_arr = np.array(q_weighted, dtype=float)
        l_arr = np.array(l_weighted, dtype=float)
        if len(q_arr) == 0 or len(l_arr) == 0:
            return None
        
        q_mz, q_int = q_arr[:, 0], q_arr[:, 1]
        l_mz, l_int = l_arr[:, 0], l_arr[:, 1]
        
        # Normalize
        q_int_norm = q_int / q_int.sum()
        l_int_norm = l_int / l_int.sum()
        
        # Match peaks
        matched_l = 0.0
        matched_q = 0.0
        matched_pairs = []
        used_q = np.zeros(len(q_mz), dtype=bool)
        
        for j, (lm, li) in enumerate(zip(l_mz, l_int_norm)):
            tol = lm * ppm_tol / 1e6
            diffs = np.abs(q_mz - lm)
            candidates = np.where((diffs <= tol) & ~used_q)[0]
            if len(candidates) > 0:
                best = candidates[np.argmin(diffs[candidates])]
                matched_l += li
                matched_q += q_int_norm[best]
                matched_pairs.append((q_int[best], l_int[j]))
                used_q[best] = True
        
        result = {
            'reverse_score': matched_l,
            'forward_score': matched_q,
            'fwd_minus_rev': matched_q - matched_l,
            'n_matched': len(matched_pairs)
        }
        
        # Ion ratio metrics
        if len(matched_pairs) >= 2:
            q_int_arr = np.array([p[0] for p in matched_pairs])
            l_int_arr = np.array([p[1] for p in matched_pairs])
            
            q_norm = q_int_arr / np.linalg.norm(q_int_arr)
            l_norm = l_int_arr / np.linalg.norm(l_int_arr)
            result['ratio_profile'] = np.dot(q_norm, l_norm)
            
            log_ratios = np.log2((q_int_arr + 1e-6) / (l_int_arr + 1e-6))
            result['log_ratio_var'] = np.var(log_ratios)
            result['max_deviation'] = np.max(np.abs(log_ratios))
            
            if len(matched_pairs) >= 3:
                spear, _ = spearmanr(rankdata(q_int_arr), rankdata(l_int_arr))
                result['spearman'] = spear if not np.isnan(spear) else -1.0
            else:
                result['spearman'] = np.nan
        else:
            result['ratio_profile'] = np.nan
            result['log_ratio_var'] = np.nan
            result['spearman'] = np.nan
            result['max_deviation'] = np.nan
        
        return result
    except:
        return None

# Compute for all deduped hits
print('Computing forward/reverse/ion-ratio metrics...')
rows = []
for idx, row in joint_dedup.iterrows():
    lib_peaks = lib_cache.get(row['library_wiki_id'])
    qry_peaks = query_cache.get(row['wiki_id'])
    scores = compute_scores(qry_peaks, lib_peaks) if (lib_peaks and qry_peaks) else None
    
    result_row = {
        'wiki_id': row['wiki_id'],
        'label': row['label'],
        'holdout': row['holdout'],
        'anno_ik14': row['anno_ik14'],
        'hit_ik14': row['hit_ik14'],
        'hit_correct': row['hit_correct'],
        'is_anno_hit': row['is_anno_hit'],
        'entropy_similarity': row['entropy_similarity'],
        'db': row['db'],
        'lib_name': row['lib_name'],
    }
    
    if scores:
        result_row.update(scores)
    else:
        for k in ['reverse_score', 'forward_score', 'fwd_minus_rev', 'ratio_profile', 
                   'log_ratio_var', 'spearman', 'max_deviation', 'n_matched']:
            result_row[k] = np.nan
    
    rows.append(result_row)

results = pd.DataFrame(rows)
valid = results['reverse_score'].notna()
print(f'Computed: {valid.sum():,}/{len(results):,} ({valid.sum()/len(results):.1%})')
print(f'  Correct hits with scores: {results[valid]["hit_correct"].sum():,}')
print(f'  Annotation hits with scores: {results[valid]["is_anno_hit"].sum():,}')

In [ ]:
# ── Hit-level AUC: correct vs incorrect (by IK14) ──
print('='*60)
print('HIT-LEVEL AUC (correct vs incorrect by IK14 match)')
print('='*60)

# Only hits with both IK14s available, non-holdout
train_h = results[
    ~results['holdout'] & 
    results['reverse_score'].notna() &
    (results['hit_ik14'] != '') & 
    (results['anno_ik14'] != '')
].copy()
y_h = train_h['hit_correct'].astype(int)  # 1=correct, 0=incorrect

print(f'Hits: {len(train_h):,} (correct={y_h.sum():,}, incorrect={(~train_h["hit_correct"]).sum():,})')

for metric in ['entropy_similarity','reverse_score','forward_score','ratio_profile','spearman']:
    v = train_h[metric].notna()
    if v.sum() < 10:
        continue
    auc = roc_auc_score(y_h[v], train_h[metric][v])
    print(f'  {metric:25s}  AUC={auc:.3f}  correct_mean={train_h[v & train_h["hit_correct"]][metric].mean():.3f}  '
          f'incorrect_mean={train_h[v & ~train_h["hit_correct"]][metric].mean():.3f}')

# ── Spectrum-level: annotation hit scores for TP vs FP ──
print('\n' + '='*60)
print('SPECTRUM-LEVEL AUC (annotation hit: TP vs FP)')
print('='*60)

# For each spectrum, get the ANNOTATION hit (IK14 match), not just the best hit
anno_hits = results[results['is_anno_hit'] & results['reverse_score'].notna()].copy()
anno_spec = (
    anno_hits
    .sort_values('entropy_similarity', ascending=False)
    .drop_duplicates(subset='wiki_id')
    .reset_index(drop=True)
)

train_s = anno_spec[~anno_spec['holdout']].copy()
y_s = (train_s['label'] == 'TP').astype(int)  # 1=TP, 0=FP

print(f'Spectra with annotation hit: {len(train_s):,} (TP={y_s.sum():,}, FP={(~y_s.astype(bool)).sum():,})')

for metric in ['entropy_similarity','reverse_score','forward_score','ratio_profile','spearman']:
    v = train_s[metric].notna()
    if v.sum() < 10:
        continue
    auc = roc_auc_score(y_s[v], train_s[metric][v])
    tp_m = train_s[v & (train_s['label']=='TP')][metric].mean()
    fp_m = train_s[v & (train_s['label']=='FP')][metric].mean()
    print(f'  {metric:25s}  AUC={auc:.3f}  TP_mean={tp_m:.3f}  FP_mean={fp_m:.3f}')

# ── Also show best-hit per spectrum (regardless of annotation match) ──
print('\n' + '='*60)
print('SPECTRUM-LEVEL AUC (best hit per spectrum: TP vs FP)')
print('='*60)

best_hits = (
    results[results['reverse_score'].notna()]
    .sort_values('entropy_similarity', ascending=False)
    .drop_duplicates(subset='wiki_id')
    .reset_index(drop=True)
)

train_b = best_hits[~best_hits['holdout']].copy()
y_b = (train_b['label'] == 'TP').astype(int)

print(f'Spectra: {len(train_b):,} (TP={y_b.sum():,}, FP={(~y_b.astype(bool)).sum():,})')

for metric in ['entropy_similarity','reverse_score','forward_score','ratio_profile','spearman']:
    v = train_b[metric].notna()
    if v.sum() < 10:
        continue
    auc = roc_auc_score(y_b[v], train_b[metric][v])
    tp_m = train_b[v & (train_b['label']=='TP')][metric].mean()
    fp_m = train_b[v & (train_b['label']=='FP')][metric].mean()
    print(f'  {metric:25s}  AUC={auc:.3f}  TP_mean={tp_m:.3f}  FP_mean={fp_m:.3f}')

In [13]:
# Save results
os.makedirs('results/reverse_similarity', exist_ok=True)
results.to_csv('results/reverse_similarity/hit_level_scores.csv', index=False)
spec_level.to_csv('results/reverse_similarity/spectrum_level_scores.csv', index=False)

print('✓ Saved:')
print(f'  results/reverse_similarity/hit_level_scores.csv ({len(results):,} rows)')
print(f'  results/reverse_similarity/spectrum_level_scores.csv ({len(spec_level):,} rows)')

print('\n' + '='*70)
print('SUMMARY: CORRECTED FORWARD/REVERSE SIMILARITY ANALYSIS')
print('='*70)
print('\nCORRECTIONS APPLIED:')
print('  [1] Removed 0.75 threshold (no survivor bias)         ✓')
print('  [2] Added entropy weighting to forward/reverse       ✓')
print('  [3] Verified entropy formula understanding           ✓')
print('  [4] Included MB-EU (public database)                 ✓')
print('  [5] Deduplicated by (wiki_id, hit_ik14)              ✓')
print(f'\nDATA SUMMARY:')
print(f'  Raw hits: 120,828  →  Filtered: {len(joint_f):,}  →  Deduplicated: {len(joint_dedup):,}')
print(f'  With valid scores: {valid.sum():,}  →  Spectrum-level: {len(spec_level):,}')
print('='*70)

✓ Saved:
  results/reverse_similarity/hit_level_scores.csv (14,002 rows)
  results/reverse_similarity/spectrum_level_scores.csv (1,510 rows)

SUMMARY: CORRECTED FORWARD/REVERSE SIMILARITY ANALYSIS

CORRECTIONS APPLIED:
  [1] Removed 0.75 threshold (no survivor bias)         ✓
  [2] Added entropy weighting to forward/reverse       ✓
  [3] Verified entropy formula understanding           ✓
  [4] Included MB-EU (public database)                 ✓
  [5] Deduplicated by (wiki_id, hit_ik14)              ✓

DATA SUMMARY:
  Raw hits: 120,828  →  Filtered: 107,875  →  Deduplicated: 14,002
  With valid scores: 1,202  →  Spectrum-level: 1,510
